# Replication: Jegadeesh-Titman (1993) Momentum
### optlab-research Â· Week 4 Validation

**Hypothesis:** Stocks with high prior 12-month returns (skipping the most recent month)
continue to outperform stocks with low prior returns over the next month.

| | |
|---|---|
| **Signal** | `momentum_12_2` â€” 12-month cumulative return, skip 1 month (tâˆ’13 to tâˆ’2) |
| **Universe** | `russell3000` (~3 000 US common stocks, approx.) |
| **Portfolio** | Equal-weight quintile long-short â€” Q5 (winners) long, Q1 (losers) short |
| **Rebalance** | Monthly, month-end |

**Literature:**
Jegadeesh & Titman (1993) *J. Finance* 48(1); Carhart (1997) *J. Finance* 52(1);
Asness, Moskowitz & Pedersen (2013) *J. Finance* 68(3).


In [ ]:
from __future__ import annotations
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
%matplotlib inline

import datetime as dt
import time
import duckdb
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display

from optlab_research.backtest.engine import Backtest, BacktestConfig
from optlab_research.signals.compute import compute_signal

# â”€â”€ Open DuckDB connection â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# WHY _CON_KEEPER:
#   optlab.db.connect() returns a _ManagedConnection context-manager wrapper.
#   The wrapper's __del__ / close logic kills the underlying DuckDB connection
#   when the object is garbage-collected.  If we put the wrapper inside a helper
#   function, it goes out of scope on return and Python GC's it immediately,
#   giving "Connection already closed" on the very next cell.
#
#   Fix: assign the wrapper to a module-level variable (_CON_KEEPER) that lives
#   in the Jupyter namespace for the entire session â€” GC can never collect it.

try:
    from optlab_research import open_connection as _get_con
except (ImportError, AttributeError):
    from optlab.db import connect as _get_con

_CON_KEEPER = _get_con()   # stays alive in notebook namespace; never GC'd

if isinstance(_CON_KEEPER, duckdb.DuckDBPyConnection):
    con = _CON_KEEPER                 # already the right type â€” use directly
elif hasattr(_CON_KEEPER, '__enter__'):
    con = _CON_KEEPER.__enter__()     # _CON_KEEPER alive above keeps this open
else:
    for _attr in ('_con', 'con', 'connection', '_connection', 'db', '_db'):
        _cand = getattr(_CON_KEEPER, _attr, None)
        if isinstance(_cand, duckdb.DuckDBPyConnection):
            con = _cand
            break
    else:
        raise RuntimeError(
            f"Cannot get a DuckDB connection from {type(_CON_KEEPER)}.\n"
            f"Attrs: {[a for a in dir(_CON_KEEPER) if not a.startswith('__')]}\n"
            "Adjust the block above to match your optlab version."
        )

print(f"DuckDB connection ready: {type(con).__name__}")


In [ ]:
crsp_info = con.execute(
    "SELECT MIN(CAST(date AS DATE)) first_date, "
    "       MAX(CAST(date AS DATE)) last_date, "
    "       COUNT(DISTINCT CAST(date AS DATE)) n_months "
    "FROM crsp_msf"
).df()
display(crsp_info)
CRSP_FIRST = str(crsp_info["first_date"].iloc[0])[:10]
CRSP_LAST  = str(crsp_info["last_date"].iloc[0])[:10]
print(f"CRSP monthly coverage: {CRSP_FIRST}  ->  {CRSP_LAST}")
# Momentum needs 12 months of history before the first signal date.
# Safe start: first_date + 13 months, but the engine handles this automatically â€”
# stocks with insufficient history get null signal_value and are excluded from the sort.


## 1. Signal validation

Compute `momentum_12_2` on a single date before running the full backtest.

**Expected output:**
- 2 500â€“3 500 valid observations (fewer for older dates)
- Right-skewed distribution centered near 0â€“15% (equity upward drift)
- Null rate 5â€“15%: stocks that lacked 10 of 12 required monthly returns
- Quintile 5 count â‰ˆ quintile 1 count â‰ˆ 500â€“700 names


In [ ]:
CHECK_DATE = "2023-06-30"  # adjust if your CRSP data ends earlier

print(f"Computing momentum_12_2 as of {CHECK_DATE} ...")
try:
    sig = compute_signal("momentum_12_2", CHECK_DATE, con, n_quantiles=5)
    null_rate = sig["signal_value"].null_count() / sig.height
    print(f"  Rows   : {sig.height:,}")
    print(f"  Nulls  : {sig['signal_value'].null_count():,}  ({null_rate:.1%})")
    print()
    print("Quintile counts:")
    display(sig.group_by("signal_quantile").len().sort("signal_quantile").to_pandas())
    print()
    print("Descriptive stats (signal_value = 12-2 cumulative return):")
    display(sig["signal_value"].describe().to_pandas())
except Exception as exc:
    print(f"SIGNAL FAILED: {exc}")
    print("Check that optlab_research/signals/library/momentum.py is importable.")
    raise

if null_rate > 0.25:
    print("WARNING: >25% null rate. Verify momentum.py lookback window (12 months) "
          "and min_obs threshold (default 10 of 12).")


In [ ]:
vals = sig["signal_value"].drop_nulls().to_numpy()
fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(vals, bins=80, color="steelblue", alpha=0.75, edgecolor="none")
ax.axvline(float(np.median(vals)), color="firebrick", lw=1.8, ls="--",
           label=f"Median: {np.median(vals):.1%}")
ax.axvline(0, color="black", lw=0.8, alpha=0.4)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_xlabel("12-2 Momentum (cumulative return)", fontsize=10)
ax.set_ylabel("Count", fontsize=10)
ax.set_title(f"momentum_12_2 cross-section -- {CHECK_DATE}", fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 2. Backtest configuration

**Methodological differences from JT (1993) to document:**

| Parameter | This backtest | JT (1993) | Expected impact |
|---|---|---|---|
| Start date | ~1990 (CRSP-limited) | 1965 | Lower Sharpe â€” misses the "golden" 1965-99 era |
| Universe | Russell 3000 (incl. NASDAQ) | NYSE/AMEX only | Higher noise, similar mean |
| Weighting | Equal-weight | Value-weight | Higher raw return, higher turnover |
| Formation | 12 months, skip 1 | 6 or 12 months | Standard modern convention |
| Holding | 1 month, monthly rebalance | 3/6/12 months tested | Shorter = more responsive |

Adjust `BACKTEST_START` to `"1965-01-01"` if your CRSP data goes that far.


In [ ]:
# Adjust BACKTEST_START to the earliest date in your CRSP file for the widest sample.
BACKTEST_START = "2019-01-01"  # SHORT WINDOW for validation; change to "1990-01-01" for full run
BACKTEST_END   = "2023-12-31"  # SHORT WINDOW for validation; change to "2024-12-31" for full run

cfg_mom = BacktestConfig(
    signal         = "momentum_12_2",
    start          = BACKTEST_START,
    end            = BACKTEST_END,
    universe       = "russell3000",
    portfolio      = "quintile_long_short",
    weighting      = "equal",
    n_quantiles    = 5,
    long_quantile  = 5,   # Q5 = winners (highest prior return) = long
    short_quantile = 1,   # Q1 = losers  (lowest  prior return) = short
)

print(f"Signal    : {cfg_mom.signal}")
print(f"Period    : {cfg_mom.start}  ->  {cfg_mom.end}")
print(f"Universe  : {cfg_mom.universe}")
print(f"Portfolio : {cfg_mom.portfolio}  |  {cfg_mom.weighting}-weight")
print(f"Long Q{cfg_mom.long_quantile}  |  Short Q{cfg_mom.short_quantile}")
print()
print("Estimated runtime: 5-20 min depending on WRDS network latency and universe size.")


In [ ]:
t0 = time.time()
print("Running momentum backtest ...")
result_mom = Backtest(cfg_mom).run(con)
elapsed = (time.time() - t0) / 60
print(f"Done in {elapsed:.1f} min  ({result_mom.returns.height} monthly return periods)")


## 3. Performance summary

In [ ]:
def _show_summary(result):
    s = result.summary()
    rows = [
        ("Period",                f"{s['start']} -> {s['end']}"),
        ("Months",                s["n_months"]),
        ("Avg N (long / short)",  f"{s['avg_n_long']:.0f} / {s['avg_n_short']:.0f}"),
        ("Ann. Return  L/S",      f"{s['ann_return_ls']:.2%}"        if s["ann_return_ls"]        else "N/A"),
        ("Ann. Vol     L/S",      f"{s['ann_vol_ls']:.2%}"           if s["ann_vol_ls"]           else "N/A"),
        ("Sharpe       L/S",      f"{s['sharpe_ls']:.3f}"            if s["sharpe_ls"]            else "N/A"),
        ("Max Drawdown L/S",      f"{s['max_drawdown_ls']:.2%}"      if s["max_drawdown_ls"]      else "N/A"),
        ("Monthly Win Rate",      f"{s['win_rate_ls']:.2%}"          if s["win_rate_ls"]          else "N/A"),
        ("Ann. Return  Long leg", f"{s['ann_return_long']:.2%}"      if s["ann_return_long"]      else "N/A"),
        ("Ann. Return  Short leg",f"{s['ann_return_short']:.2%}"     if s["ann_return_short"]     else "N/A"),
        ("Avg Monthly Turnover",  f"{s['avg_monthly_turnover']:.2%}" if s["avg_monthly_turnover"] else "N/A"),
    ]
    display(pd.DataFrame(rows, columns=["Metric", "Value"]).set_index("Metric"))

_show_summary(result_mom)

## 4. Benchmark comparison

Benchmarks are calibrated to **equal-weight quintile L/S, monthly rebalance,
US broad market, post-1990**. The original JT (1993) results (1965-1989) are
substantially stronger; the ranges below account for the post-2000 period
which includes major momentum crashes in 2001, 2009, and 2020.

`PASS` = actual is within the acceptable range.
`CHECK` = outside range â€” investigate before drawing research conclusions.


In [ ]:
def _bm_table(result, benchmarks):
    stats = result.summary()
    rows = []
    for key, info in benchmarks.items():
        actual   = stats.get(key)
        lo, hi   = info.get("range", (None, None))
        expected = info.get("expected")
        if actual is not None and lo is not None:
            within = lo <= actual <= hi
            pct    = (actual - expected) / abs(expected) * 100 if expected else None
            flag   = "PASS" if within else "CHECK"
        else:
            pct, flag = None, "N/A"
        rows.append({
            "Metric":    info.get("label", key),
            "Acceptable range": (f"[{lo:.3f}, {hi:.3f}]" if lo is not None else "--"),
            "Actual":    (f"{actual:.3f}" if actual is not None else "N/A"),
            "Ref (mid)": (f"{expected:.3f}" if expected is not None else "--"),
            "% diff":    (f"{pct:+.1f}%" if pct is not None else "--"),
            "Result":    flag,
            "Source":    info.get("source", ""),
        })
    return pd.DataFrame(rows).set_index("Metric")

MOM_BENCHMARKS = {
    "ann_return_ls": {
        "label"   : "Ann. Return (L/S)",
        "expected": 0.07,
        "range"   : (0.02, 0.14),
        "source"  : "Carhart (1997); AQR UMD post-1990",
    },
    "sharpe_ls": {
        "label"   : "Sharpe (L/S)",
        "expected": 0.40,
        "range"   : (0.10, 0.70),
        "source"  : "Asness, Moskowitz & Pedersen (2013) extended sample",
    },
    "ann_vol_ls": {
        "label"   : "Ann. Vol (L/S)",
        "expected": 0.15,
        "range"   : (0.09, 0.22),
        "source"  : "AQR UMD monthly returns, post-1990",
    },
    "win_rate_ls": {
        "label"   : "Monthly Win Rate",
        "expected": 0.58,
        "range"   : (0.50, 0.66),
        "source"  : "Jegadeesh & Titman (1993) Table 1",
    },
    "max_drawdown_ls": {
        "label"   : "Max Drawdown (L/S)",
        "expected": -0.55,
        "range"   : (-0.99, -0.25),  # momentum crashes hard; wide range
        "source"  : "Known: momentum crashed ~80% in Mar-May 2009",
    },
}

display(_bm_table(result_mom, MOM_BENCHMARKS))


In [ ]:
fig = result_mom.plot_cumulative(figsize=(13, 5))
plt.show()


In [ ]:
fig = result_mom.plot_drawdown(figsize=(13, 3))
plt.show()


## 5. Seasonality: the January effect

Momentum is documented to **reverse in January** (Jegadeesh & Titman 2001;
Grundy & Martin 2001). Losers (the short side) bounce in January due to
tax-loss selling reversal. The January bar should be noticeably lower or negative.

A strongly positive January is a warning sign of a data alignment error.
A strongly negative January (âˆ’3% to âˆ’6%) is realistic and expected.


In [ ]:
df_pd = result_mom.returns.to_pandas()
df_pd["month"]   = pd.to_datetime(df_pd["date"]).dt.month
df_pd["mon_nm"]  = pd.to_datetime(df_pd["date"]).dt.strftime("%b")

grp = df_pd.groupby("month").agg(
    mean   = ("ls_ret", "mean"),
    std    = ("ls_ret", "std"),
    n      = ("ls_ret", "count"),
    mon_nm = ("mon_nm", "first"),
).reset_index()
grp["t_stat"] = grp["mean"] / (grp["std"] / np.sqrt(grp["n"]))

colors = ["firebrick" if x < 0 else "steelblue" for x in grp["mean"]]
fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(grp["mon_nm"], grp["mean"] * 100, color=colors, alpha=0.8, edgecolor="none")
ax.axhline(0, color="black", lw=0.8)
for bar, t_val in zip(bars, grp["t_stat"]):
    y = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2,
            y + (0.05 if y >= 0 else -0.22),
            f"t={t_val:.1f}", ha="center", va="bottom", fontsize=7)
ax.set_ylabel("Avg monthly L/S return (%)", fontsize=10)
ax.set_title("momentum_12_2 -- average L/S return by calendar month", fontsize=11)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

jan_row  = grp[grp["month"] == 1].iloc[0]
ex_jan   = grp[grp["month"] != 1]["mean"].mean()
print(f"January avg L/S    : {jan_row['mean']:.2%}  (t = {jan_row['t_stat']:.2f})")
print(f"Ex-January avg L/S : {ex_jan:.2%}")
print(f"January drag vs avg: {jan_row['mean'] - ex_jan:.2%}")


In [ ]:
# 36-month rolling annualized Sharpe -- shows time-variation in momentum profitability
rets_s      = result_mom.returns.to_pandas().set_index("date")["ls_ret"].dropna()
roll_mean   = rets_s.rolling(36).mean() * 12
roll_std    = rets_s.rolling(36).std()  * np.sqrt(12)
roll_sharpe = roll_mean / roll_std

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(roll_sharpe.index, roll_sharpe, color="steelblue", lw=1.5,
        label="36-month rolling Sharpe")
ax.axhline(0, color="black", lw=0.8, ls="--")
full_mean = float(roll_sharpe.dropna().mean())
ax.axhline(full_mean, color="steelblue", lw=1.0, ls=":",
           alpha=0.7, label=f"Full-period mean: {full_mean:.2f}")
ax.fill_between(roll_sharpe.index, roll_sharpe, 0,
                where=(roll_sharpe < 0), color="firebrick", alpha=0.2,
                label="Negative Sharpe periods")
ax.set_ylabel("Rolling Sharpe (annualized)", fontsize=10)
ax.set_title("momentum_12_2 -- 36-month rolling Sharpe  (momentum crashes visible)", fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Quintile spread check

Plot each quintile's long-only cumulative return. The spread should be monotone:
Q5 > Q4 > Q3 > Q2 > Q1 (by annualized return). A non-monotone pattern â€” e.g.,
Q3 outperforming Q5 â€” is a signal construction red flag, not just a noisy sample.


In [ ]:
# â”€â”€ Quintile spread diagnostic (optional â€” set True to run) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Runs 5 additional long-only backtests to confirm monotone Q5>Q4>Q3>Q2>Q1 spread.
# Each takes 5-15 min; total ~50 min extra. Leave False for a quick first pass.
RUN_QUINTILE_SPREAD = False

if not RUN_QUINTILE_SPREAD:
    print("Quintile spread check skipped. Set RUN_QUINTILE_SPREAD = True to run.")
    print("Expected: annualized return is monotone Q5 > Q4 > Q3 > Q2 > Q1.")
else:
    quintile_series = {}
    for q in range(1, 6):
        cfg_q = BacktestConfig(
            signal         = "momentum_12_2",
            start          = BACKTEST_START,
            end            = BACKTEST_END,
            universe       = "russell3000",
            portfolio      = "long_only",
            weighting      = "equal",
            n_quantiles    = 5,
            long_quantile  = q,
            short_quantile = 1,   # unused for long_only but required
        )
        r_q = Backtest(cfg_q).run(con)
        ser = r_q.returns.to_pandas().set_index("date")["long_ret"]
        quintile_series[f"Q{q}"] = ser
        sq  = r_q.summary()
        print(f"  Q{q}: ann. return = {sq['ann_return_long']:.2%}  Sharpe = {sq['sharpe_long']:.3f}")

    fig, ax = plt.subplots(figsize=(13, 5))
    palette = ["#d62728", "#ff7f0e", "#8c8c8c", "#2ca02c", "#1f77b4"]  # red -> blue
    for (label, ser), color in zip(quintile_series.items(), palette):
        cum   = (1 + ser.fillna(0)).cumprod() - 1
        ann_r = float((1 + ser.dropna()).prod() ** (12 / max(len(ser.dropna()), 1)) - 1)
        lw    = 2.0 if label in ("Q1", "Q5") else 1.0
        ax.plot(cum.index, cum, label=f"{label}  ({ann_r:.1%} ann.)", color=color, lw=lw)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
    ax.set_title("momentum_12_2 -- cumulative return by quintile (monotone Q5>...>Q1?)", fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


## 7. Known discrepancies from published results

| Source of difference | Direction | Magnitude |
|---|---|---|
| Sample starts ~1990, not 1965 | Lower Sharpe (2001, 2009 crashes in sample; golden 1965-99 missing) | Moderate-Large |
| Universe includes NASDAQ | More small-cap noise | Small |
| Equal-weight vs. value-weight | Higher raw return, higher turnover, more small-cap tilt | Small-Moderate |
| Monthly hold (1-month); JT also tested 3/6/12-month holds | Different from JT Table 1 row averages | Varies |
| No transaction costs (v0) | Overstated net-of-cost performance | Meaningful at high turnover |

**Momentum crashes to expect in your sample:**
- 2001-2002: growthâ†’value rotation crushes momentum
- 2009 Mar-May: sharp market recovery wipes out short book
- 2020 Mar-Apr: COVID crash reversal

If `max_drawdown_ls` is between -45% and -85%, that is correct behavior â€” not a bug.


In [ ]:
SAVE_PATH = "outputs/01_momentum_replication"
result_mom.save(SAVE_PATH)
print(f"Results saved: {SAVE_PATH}/")
print("  returns.parquet, holdings_summary.parquet, monthly_turnover.parquet, manifest.json")
print()
print("Week 5: re-run with tcost_bps=20 in BacktestConfig to measure Sharpe degradation.")
print("Week 6: load returns.parquet and run FF6 attribution.")
print("        Verify: UMD loading ~1.0, alpha near zero after controlling for UMD.")


## Conclusion

**Fill in after running:**

| Metric | This run | JT (1993) benchmark | Within range? |
|---|---|---|---|
| Ann. Return L/S | | 8-10% | |
| Sharpe L/S | | 0.40-0.55 | |
| Max Drawdown | | -50% to -80% | |
| January avg L/S | | Negative | |
| Monthly win rate | | ~58% | |

Next steps:
- **Week 5:** Add `tcost_bps=20`. Expected: Sharpe degrades ~30-50% for momentum (high turnover).
- **Week 6:** FF6 attribution. Expected: UMD beta â‰ˆ 1.0, alpha â‰ˆ 0 (we ARE the factor).
